![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/image/DolphinForDocumentParsing_Scanned_Document_Intelligence.ipynb)

# Introducing DolphinForDocumentParsing in SparkNLP

This notebook showcases the newly added `DolphinForDocumentParsing` annotator, which runs
ByteDance's [Dolphin-1.5](https://huggingface.co/ByteDance/Dolphin-1.5) to parse document images
into structured elements: paragraphs as text, tables as HTML, formulas as LaTeX and code as code,
each with its reading order and bounding box.

A scanned PDF has no text layer, so Spark NLP's readers get nothing from it. We will scan a few
pages of *Attention Is All You Need* ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)), parse
them with Dolphin, query their tables with SQL and search them with BGE embeddings.

## Setup and Initialization

Let's keep in mind a few things before we start 😊

- `DolphinForDocumentParsing` was introduced in Spark NLP 7.0.1. Please make sure you have
  upgraded to the latest Spark NLP release.
- We also install `pymupdf` to render the PDF pages.

In [ ]:
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash
! pip install -q pyspark==3.5.0 pymupdf

Let's start Spark with Spark NLP included via our simple `start()` function

In [ ]:
import sparknlp

# let's start Spark with Spark NLP
spark = sparknlp.start(memory="6G")

## Loading the model

Let's download the pretrained `dolphin_1_5` model (about 1.3 GB). We load it once and switch its
mode with setters later on. To import other Dolphin checkpoints, see the
[import notebook](../../transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_DolphinForDocumentParsing.ipynb).

In [ ]:
from sparknlp.annotator import DolphinForDocumentParsing

dolphin = DolphinForDocumentParsing.pretrained("dolphin_1_5", "en") \
    .setInputCols(["image"]) \
    .setOutputCol("elements") \
    .setParsingMode("page")

## Making a scanned PDF

We render five pages of the paper at 200 DPI into an image-only PDF, which is what a scanner
produces: the pixels survive, the text layer does not. The pages cover the title page, the
formulas, Figure 2 and both tables.

In [ ]:
import urllib.request
import pymupdf
from PIL import Image

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762v7", "attention.pdf")

PAGES = [1, 4, 5, 6, 8]
paper = pymupdf.open("attention.pdf")

def image_only_pdf(path, dpi):
    out = pymupdf.open()
    for p in PAGES:
        pix = paper[p - 1].get_pixmap(dpi=dpi)
        page = out.new_page(width=paper[p - 1].rect.width, height=paper[p - 1].rect.height)
        page.insert_image(page.rect, stream=pix.tobytes("jpeg", jpg_quality=85))
    out.save(path)
    return pix

image_only_pdf("attention_scanned.pdf", dpi=200)
scans = []
for p in PAGES:
    pix = paper[p - 1].get_pixmap(dpi=200)
    scans.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))

scanned = pymupdf.open("attention_scanned.pdf")
print("text layer, original pages:", sum(len(paper[p - 1].get_text()) for p in PAGES), "chars")
print("text layer, scanned pages: ", sum(len(page.get_text()) for page in scanned), "chars")
print("page image:", scans[0].size)

## Reading the scan with the PDF reader

The PDF reader extracts the text layer, so it reads the original pages and finds nothing on the
scan:

In [ ]:
from pyspark.sql import functions as F
from sparknlp.reader import SparkNLPReader

def text_layer(path, pages):
    return SparkNLPReader(spark).pdf(path) \
        .selectExpr("explode(pdf) as e") \
        .select(F.col("e.metadata.pageNumber").cast("int").alias("page"), "e.content") \
        .where(F.col("page").isin(pages)) \
        .groupBy("page") \
        .agg(F.concat_ws(" ", F.collect_list("content")).alias("text")) \
        .select("page", F.length("text").alias("chars"), F.substring("text", 1, 70).alias("text")) \
        .orderBy("page")

print("original")
text_layer("attention.pdf", PAGES).show(truncate=False)
print("scan")
text_layer("attention_scanned.pdf", list(range(1, len(PAGES) + 1))).show(truncate=False)

## Parsing the scan with Dolphin

`Reader2Image` turns each PDF page into an image. In `page` mode Dolphin first finds every
element's box, label and reading order, then transcribes each element with the prompt for its
type. Every element becomes its own annotation with `dolphinLabel`, `elementType`,
`readingOrder`, `bbox` and `pageNumber` metadata, and tables also get `tableJson`.

Parsing is the slow step, so we run it once and save the result as Parquet.

In [ ]:
import time
from pyspark.ml import Pipeline
from sparknlp.reader.reader2image import Reader2Image

empty_df = spark.createDataFrame([], "string").toDF("text")

reader = Reader2Image() \
    .setContentType("application/pdf") \
    .setContentPath("attention_scanned.pdf") \
    .setOutputCol("image")

parse_pipeline = Pipeline(stages=[reader, dolphin]).fit(empty_df)

# map the scan's page numbers back to the paper's
paper_page = F.element_at(F.array(*[F.lit(p) for p in PAGES]), F.col("page"))

start = time.time()
parse_pipeline.transform(empty_df) \
    .selectExpr("explode(elements) as e") \
    .selectExpr(
        "cast(e.metadata['pageNumber'] as int) as page",
        "cast(e.metadata['readingOrder'] as int) as reading_order",
        "e.metadata['dolphinLabel'] as label",
        "e.metadata['elementType'] as element_type",
        "e.metadata['bbox'] as bbox",
        "e.metadata['tableJson'] as table_json",
        "e.result as text") \
    .withColumn("paper_page", paper_page) \
    .write.mode("overwrite").parquet("attention_elements.parquet")
page_mode_seconds = time.time() - start

elements = spark.read.parquet("attention_elements.parquet").cache()
print(f"{elements.count()} elements from {len(PAGES)} pages in {page_mode_seconds:.0f}s")

Element labels per page. Dolphin also marks page furniture (`header`, `foot`, `watermark`), which
a text layer can't tell apart from content:

In [ ]:
elements.groupBy("label").pivot("paper_page", PAGES).count().na.fill(0).orderBy("label").show()

The title page, in reading order:

In [ ]:
elements.where("paper_page = 1").orderBy("reading_order") \
    .select("reading_order", "label", "text").show(truncate=90)

### See what Dolphin saw

`bbox` is in the page image's pixels, so we can draw it on the scan:

In [ ]:
from PIL import ImageDraw, ImageFont

COLORS = {"tab": "#d62728", "equ": "#9467bd", "fig": "#2ca02c", "cap": "#ff7f0e",
          "para": "#1f77b4", "half_para": "#1f77b4", "list": "#1f77b4", "fnote": "#8c564b"}
FONT = ImageFont.load_default(size=24)

def draw_page(paper_page_no, highlight=None, width=800):
    image = scans[PAGES.index(paper_page_no)].convert("RGBA")
    overlay = Image.new("RGBA", image.size)
    draw = ImageDraw.Draw(overlay)
    if highlight:
        draw.rectangle(list(map(int, highlight.split(","))), fill=(255, 200, 0, 100))
    for row in elements.where(F.col("paper_page") == paper_page_no).collect():
        x1, y1, x2, y2 = map(int, row.bbox.split(","))
        color = COLORS.get(row.label, "#7f7f7f")
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        tag = f"{row.reading_order} {row.label}"
        _, _, right, bottom = FONT.getbbox(tag)
        draw.rectangle([x1, y1 - bottom - 8, x1 + right + 8, y1], fill=color)
        draw.text((x1 + 4, y1 - bottom - 4), tag, fill="white", font=FONT)
    page = Image.alpha_composite(image, overlay).convert("RGB")
    return page.resize((width, int(page.height * width / page.width)))

draw_page(4)

## Tables as DataFrames

Tables carry `tableJson` in the same shape `Reader2Table` produces for digital documents.
Captions are separate elements, so we join each table to the caption next to it:

In [ ]:
TABLE_SCHEMA = "struct<caption:string,header:array<string>,rows:array<array<string>>>"

tables = elements.where("label = 'tab'").alias("t") \
    .join(elements.where("label = 'cap'").alias("c"),
          (F.col("t.page") == F.col("c.page")) &
          (F.abs(F.col("t.reading_order") - F.col("c.reading_order")) == 1), "left") \
    .select(F.col("t.paper_page"),
            F.col("c.text").alias("caption"),
            F.from_json("t.table_json", TABLE_SCHEMA).alias("table"))

tables.select("paper_page", F.substring("caption", 1, 80).alias("caption"),
              F.size("table.rows").alias("rows")).show(truncate=False)

Table 1 has one header row, so its first row names the columns:

In [ ]:
header, *rows = tables.where("caption like 'Table 1%'").first().table.rows
spark.createDataFrame(rows, header).show(truncate=False)

Table 2 has a two-row header, and `tableJson` keeps the cell text but not the spans, so we name
the five columns ourselves:

In [ ]:
top, sub, *rows = tables.where("caption like 'Table 2%'").first().table.rows
print(top)
print(sub)

columns = ["model", "bleu_en_de", "bleu_en_fr", "flops_en_de", "flops_en_fr"]
spark.createDataFrame(rows, columns).createOrReplaceTempView("table_2")

The training costs come back as LaTeX (`2.3 $\cdot$ 10$^{19}$`), which SQL can turn back into
numbers:

In [ ]:
spark.sql(r"""
CREATE OR REPLACE TEMP VIEW wmt14 AS
SELECT model,
       CAST(NULLIF(bleu_en_de, '') AS DOUBLE) AS bleu_en_de,
       CAST(NULLIF(bleu_en_fr, '') AS DOUBLE) AS bleu_en_fr,
       CAST(regexp_extract(flops_en_de, '^([0-9.]+)', 1) AS DOUBLE)
         * pow(10, CAST(regexp_extract(flops_en_de, '\\{([0-9]+)\\}', 1) AS INT)) AS flops_en_de
FROM table_2
""")

spark.sql("""
SELECT model, bleu_en_de, flops_en_de,
       round(flops_en_de / (SELECT flops_en_de FROM wmt14 WHERE model = 'Transformer (big)'), 1)
         AS cost_vs_big
FROM wmt14
WHERE bleu_en_de IS NOT NULL
ORDER BY bleu_en_de DESC
""").show(truncate=False)

## Formulas as LaTeX

Display formulas are labelled `equ` and come back as LaTeX:

In [ ]:
from IPython.display import display, Math

for row in elements.where("label = 'equ'").orderBy("page", "reading_order").collect():
    print(f"p. {row.paper_page}: {row.text}")
    display(Math(row.text))

Inline math in paragraphs comes back as LaTeX too:

In [ ]:
print(elements.where("label = 'para' and text like '%sqrt%'").orderBy("page", "reading_order").first().text)

## Searching the scan

Each element is a natural retrieval chunk, and it keeps its page and box, so an answer can point
at the region of the scan it came from. We drop page furniture, prefix each chunk with its section
heading and embed the chunks with BGE:

In [ ]:
from pyspark.sql import Window
from sparknlp.base import DocumentAssembler, EmbeddingsFinisher
from sparknlp.annotator import BGEEmbeddings

so_far = Window.orderBy("page", "reading_order").rowsBetween(Window.unboundedPreceding, 0)
heading = F.last(F.when(F.col("label").startswith("sec"), F.col("text")), ignorenulls=True)

chunks = elements \
    .withColumn("section", F.coalesce(heading.over(so_far), F.lit(""))) \
    .where("label in ('para', 'half_para', 'list', 'tab', 'equ', 'cap', 'fnote')") \
    .withColumn("body", F.when(F.col("label") == "tab", F.regexp_replace("text", "<[^>]+>", " "))
                         .otherwise(F.col("text"))) \
    .withColumn("chunk", F.concat_ws("\n", "section", "body")) \
    .select("paper_page", "label", "bbox", "text", "chunk")

embed_pipeline = Pipeline(stages=[
    DocumentAssembler().setInputCol("chunk").setOutputCol("document"),
    BGEEmbeddings.pretrained("bge_small_en_v1.5", "en")
        .setInputCols(["document"]).setOutputCol("embeddings"),
    EmbeddingsFinisher().setInputCols(["embeddings"]).setOutputCols(["vector"]),
]).fit(chunks)

index = embed_pipeline.transform(chunks) \
    .select("paper_page", "label", "bbox", "text", F.col("vector")[0].alias("vector")) \
    .collect()
print(len(index), "chunks embedded")

In [ ]:
import numpy as np

matrix = np.array([row.vector for row in index])
matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)

def ask(question, k=3):
    q = embed_pipeline.transform(spark.createDataFrame([(question,)], ["chunk"])) \
        .select(F.col("vector")[0]).first()[0]
    scores = matrix @ (np.array(q) / np.linalg.norm(q))
    top = np.argsort(-scores)[:k]
    print(question)
    for i in top:
        row = index[i]
        snippet = " ".join(row.text.split())[:100]
        print(f"  {scores[i]:.3f}  p.{row.paper_page} {row.label:<9} {snippet}")
    print()
    return index[top[0]]

ask("How does the model know the order of the words?")
ask("Why are the dot products scaled before the softmax?")
ask("How many attention heads are used?")
bleu_hit = ask("What BLEU score does the big Transformer reach on English-to-German?")

The best hit for the last question, highlighted on the scan:

In [ ]:
draw_page(bleu_hit.paper_page, highlight=bleu_hit.bbox)

## Layout-only mode

`layout` mode finds the boxes, labels and reading order without transcribing anything, which
makes it a quick first pass to find the pages worth parsing:

In [ ]:
dolphin.setParsingMode("layout")

start = time.time()
triage = parse_pipeline.transform(empty_df) \
    .selectExpr("explode(elements) as e") \
    .selectExpr("cast(e.metadata['pageNumber'] as int) as page", "e.metadata['dolphinLabel'] as label") \
    .withColumn("paper_page", paper_page) \
    .groupBy("paper_page") \
    .agg(F.count("*").alias("elements"),
         F.sum(F.expr("int(label = 'tab')")).alias("tables"),
         F.sum(F.expr("int(label = 'equ')")).alias("formulas"),
         F.sum(F.expr("int(label = 'fig')")).alias("figures")) \
    .orderBy("paper_page") \
    .collect()
layout_seconds = time.time() - start

dolphin.setParsingMode("page")
spark.createDataFrame(triage).show()
print(f"layout only: {layout_seconds:.0f}s, full parse: {page_mode_seconds:.0f}s")